In [ ]:
from pathlib import Path as _ExperimentPath
import sys

ROOT = _ExperimentPath.cwd()
EXPERIMENT_DIR = ROOT / 'experiment/misc'
sys.path[:0] = [str(ROOT / 'src'), str(ROOT), str(ROOT / 'experiment')]


waterfall & error floor region

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({'font.size': 13, 'mathtext.fontset': 'cm', 'font.family': 'serif'})
p = np.logspace(-6, 0, 600)
omega = 0.65
C = 0.008
low_noise = C * p ** omega
x = np.log10(p)
x0 = -1.6
k = 5.5
waterfall = 0.9 / (1 + np.exp(-k * (x - x0)))
P_L = low_noise + waterfall
P_L = np.clip(P_L, 1e-07, 0.95)
p_ref = np.logspace(-5.6, -3.2, 100)
P_ref = C * p_ref ** omega

fig, ax = plt.subplots(figsize=(9, 6.8))
ax.loglog(p, P_L, linewidth=2.5)
ax.loglog(p_ref, P_ref, '--', linewidth=1.2, alpha=0.6)
ax.set_xlabel('Physical error rate $p$', fontsize=16)
ax.set_ylabel('Logical error rate $P_L$', fontsize=16)
ax.set_xlim(1e-06, 1.2)
ax.set_ylim(7e-07, 1.5)
ax.grid(True, which='major', alpha=0.2)
ax.grid(False, which='minor')
ax.text(2.8e-06, 0.00023, 'Error floor region', fontsize=18)
ax.annotate(
    '$P_L(p)\\sim A_{w_*}\\,p^{w_*}$',
    xy=(2.7e-05, 1.4e-05),
    xytext=(3.3e-06, 9.2e-05),
    fontsize=18,
    arrowprops=dict(arrowstyle='->', lw=1.5),
)
ax.annotate(
    'slope $= w_*$',
    xy=(0.00022, C * 0.00022 ** omega),
    xytext=(0.00045, 1.6e-05),
    fontsize=15,
    arrowprops=dict(arrowstyle='->', lw=1.2, alpha=0.7),
    alpha=0.75,
)
ax.annotate(
    'dominated by minimum weight errors',
    xy=(1.3e-05, C * 1.3e-05 ** omega),
    xytext=(3e-05, 1.4e-06),
    fontsize=14,
    arrowprops=dict(arrowstyle='->', lw=1.2, alpha=0.7),
    alpha=0.75,
)
ax.annotate(
    'Waterfall region',
    xy=(0.006, 0.024),
    xytext=(0.007, 0.0014),
    fontsize=20,
    arrowprops=dict(arrowstyle='->', lw=1.5),
)
plt.tight_layout()

plt.show()


triangular code

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, PathPatch
from matplotlib.path import Path

plt.rcParams.update({'font.family': 'serif', 'mathtext.fontset': 'cm', 'font.size': 14})
CHECK_COLOR = '#88ADD0'
NODE_COLOR = 'black'
NODE_SIZE = 55
LABEL_SIZE = 15

def p(x, y):
    return np.array([x, y], dtype=float)

def draw_node(
    ax,
    pos,
    label,
    label_dy=0.16,
):
    """Draw one data qubit and its index."""
    ax.scatter([pos[0]], [pos[1]], s=NODE_SIZE, color=NODE_COLOR, zorder=10)
    ax.text(
        pos[0],
        pos[1] + label_dy,
        str(label),
        ha='center',
        va='bottom',
        fontsize=LABEL_SIZE,
        color='black',
        zorder=20,
    )

def draw_triangle_check(
    ax,
    a,
    b,
    c,
):
    """
    Three-body bulk stabilizer acting on vertices a,b,c.
    """
    patch = Polygon(
        [a, b, c],
        closed=True,
        facecolor=CHECK_COLOR,
        edgecolor='none',
        zorder=1,
    )
    ax.add_patch(patch)

def draw_boundary_check(
    ax,
    top,
    bottom,
    side,
    bulge=0.62,
    tangential=0.18,
):
    """
    Draw a fuller two-body boundary stabilizer between top and bottom.

    Parameters
    ----------
    top, bottom : np.ndarray
        Endpoints of the boundary check.
    side : str
        'left' or 'right'
    bulge : float
        How far the curved boundary bulges outward.
    tangential : float
        Small tangential spread to make the cap rounder / fuller.
    """

    d = bottom - top
    d = d / np.linalg.norm(d)
    if side == 'left':
        normal = np.array([d[1], -d[0]])
    elif side == 'right':
        normal = np.array([-d[1], d[0]])
    else:
        raise ValueError("side must be 'left' or 'right'")

    normal = normal / np.linalg.norm(normal)
    c1 = bottom + bulge * normal - tangential * d
    c2 = top + bulge * normal + tangential * d
    verts = [tuple(top), tuple(bottom), tuple(c1), tuple(c2), tuple(top)]
    codes = [Path.MOVETO, Path.LINETO, Path.CURVE4, Path.CURVE4, Path.CURVE4]
    patch = PathPatch(Path(verts, codes), facecolor=CHECK_COLOR, edgecolor='none', zorder=1)
    ax.add_patch(patch)

def plot_T1():
    q1 = p(0.5, 0.0)
    q2 = p(1.5, 0.0)
    q3 = p(0.0, 1.0)
    q4 = p(1.0, 1.0)
    q5 = p(2.0, 1.0)

    fig, ax = plt.subplots(figsize=(4.3, 2.6))
    draw_boundary_check(ax, top=q3, bottom=q1, side='left')
    draw_triangle_check(ax, q1, q2, q4)
    draw_boundary_check(ax, top=q5, bottom=q2, side='right')
    nodes = {
        1: q1,
        2: q2,
        3: q3,
        4: q4,
        5: q5,
    }

    for label, pos in nodes.items():
        draw_node(ax, pos, label)
    ax.set_aspect('equal')
    ax.set_xlim(-0.55, 2.55)
    ax.set_ylim(-0.25, 1.38)
    ax.axis('off')
    plt.tight_layout()

    plt.show()

def plot_T2():
    q1 = p(1.0, 0.0)
    q2 = p(2.0, 0.0)
    q3 = p(0.5, 1.0)
    q4 = p(1.5, 1.0)
    q5 = p(2.5, 1.0)
    q6 = p(0.0, 2.0)
    q7 = p(1.0, 2.0)
    q8 = p(2.0, 2.0)
    q9 = p(3.0, 2.0)

    fig, ax = plt.subplots(figsize=(5.2, 3.6))
    draw_boundary_check(ax, top=q3, bottom=q1, side='left')
    draw_triangle_check(ax, q1, q2, q4)
    draw_boundary_check(ax, top=q5, bottom=q2, side='right')
    draw_boundary_check(ax, top=q6, bottom=q3, side='left')
    draw_triangle_check(ax, q3, q4, q7)
    draw_triangle_check(ax, q4, q5, q8)
    draw_boundary_check(ax, top=q9, bottom=q5, side='right')
    nodes = {
        1: q1,
        2: q2,
        3: q3,
        4: q4,
        5: q5,
        6: q6,
        7: q7,
        8: q8,
        9: q9,
    }

    for label, pos in nodes.items():
        draw_node(ax, pos, label)
    ax.set_aspect('equal')
    ax.set_xlim(-0.55, 3.55)
    ax.set_ylim(-0.25, 2.38)
    ax.axis('off')
    plt.tight_layout()

    plt.show()

plot_T1()
plot_T2()


In [ ]:
import stim

def z_stabilizer_measurement(data_qubits, ancilla):
    """
    Measure a Z-type stabilizer:
        Z_{data_qubits[0]} Z_{data_qubits[1]} ...

    Args:
        data_qubits: list[int]
            Data qubit indices.
        ancilla: int
            Ancilla qubit index.

    Returns:
        stim.Circuit
    """

    circuit = stim.Circuit()
    circuit.append('R', [ancilla])
    for q in data_qubits:
        circuit.append('CX', [q, ancilla])
    circuit.append('M', [ancilla])

    return circuit

circuit = z_stabilizer_measurement(data_qubits=[0, 1, 2, 3], ancilla=4)
circuit.diagram('timeline-svg')
